# Code to download and preprocess REMA strips over ice shelves

In [ ]:
project_crs = "EPSG:3031"

projectDir = "/Users/rfk471/Dropbox/REMA-strips-correction"

# This is just a name that you give your region
# This will be used when saving files and creating folders
region_id = "PIG" 

In [ ]:
import geopandas as gpd
import preprocessing_tools as ptools

## Define an Area Of Interest, and call that region something

Here I have called the region "PIG" which we will use througout the code

In [ ]:

# Coordinates of Area Of Interest (AOI)
coords = [
    (-102.53687728768465, -75.49231807240791),
    (-100.47694076424715, -75.49231807240791),
    (-100.47694076424715, -75.02383074354489),
    (-102.53687728768465, -75.02383074354489),
    (-102.53687728768465, -75.49231807240791)
]

# CRS of coordinates
coords_crs = "EPSG:4326"

region = ptools.polygon_coords_to_gdf(coords,coords_crs,project_crs,projectDir,region_id)



## Download the REMA strips

Before downloading the REMA strips you need to download the REMA Strip DEM extent index SHP file from https://www.pgc.umn.edu/data/rema/

Make sure that the shapefile gets stored here: `{projectDir}/data/initial/REMA/REMA_Strip_Index_s2s041.shp`

Now you can run the download script like this if you want to keep the original resolution:

`python download-REMA-strips.py --region_id PIG --projectDir "/path/to/project" --resample false`

Or like this if you want to resample to for instance 20 m resolution:

`python download-REMA-strips.py --region_id PIG --projectDir "/path/to/project" --resample true --resolution 20`


If you're on MacOS and want to prevent your laptop from sleeping while downloading, you can use:

`caffeinate -i python download-REMA-strips.py --region_id PIG --projectDir "/path/to/project" --resample false`

Here is the complete list of flags:

`--region_id PIG`: The name of the region/AOI

`--project_crs EPSG:3031`: CRS of the project, EPSG:3031 is the default

`--n_threads 8`: Number of parallel threads for the download, default is 8

`--projectDir "/path/to/project"`: path to project directory

`--resample true`: whether or not to resample the strips

`--resolution 20`: Resample resolution in meters if resampling is chosen

## Bit masking

Cleaning the strips based on the provided bitmask

In [ ]:
# Bit masking

n_threads = 8 # Number of CPU cores to use

ptools.bit_masking(region_id,projectDir,n_threads)

## Tide correction

The tides are being calculated on a 1x1 km grid and interpolated onto the REMA grid. The original resolution of the tide model is 2 km.

If you don't already have CATS2008_v2023, you first need to it. You can download it here: https://www.usap-dc.org/view/dataset/601772

I have chosen to save CATS in data/initial, but you can choose where you save it and just point to the right directory below with `tide_model_dir`.

In [ ]:
ptools.tide_correction(
    region_id="PIG",
    projectDir=projectDir,
    tide_model_dir=f"{projectDir}/data/initial",
    tide_resolution=1000,
    tide_buffer=5000,
    save_tide=True,
)